In [ ]:
import requests
FIELDS = ['dr_no','date_occ','time_occ','area_name','crm_cd_desc','vict_age','vict_sex','vict_descent','premis_desc','location','cross_street','lat','lon','mocodes']
BASE = 'https://data.lacity.org/resource/d5tf-ez2w.json'
print('Config OK')

In [ ]:
# Fetch with pagination - 10K per batch to avoid timeouts
all_records = []
offset = 0
batch_size = 10000
select_str = ','.join(FIELDS)
while True:
    url = BASE + '?$limit=' + str(batch_size) + '&$offset=' + str(offset) + '&$select=' + select_str + '&$where=date_occ%3E%272022-01-01T00:00:00%27&$order=date_occ%20DESC'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    batch = r.json()
    if not batch:
        break
    all_records.extend(batch)
    print(len(all_records), 'records (batch', offset // batch_size + 1, ')')
    if len(batch) < batch_size:
        break
    offset += batch_size
print('TOTAL:', len(all_records))

In [ ]:
# Normalize all values to strings
rows = [tuple(str(rec.get(f, '') or '') for f in FIELDS) for rec in all_records]
print('Normalized:', len(rows), 'rows')

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F
schema = StructType([StructField(f, StringType(), True) for f in FIELDS])
df = spark.createDataFrame(rows, schema)
print('DataFrame:', df.count(), 'rows')

In [ ]:
df2 = df.withColumn('crash_year', F.year(F.to_date('date_occ')))
df2.write.format('delta').mode('overwrite').partitionBy('crash_year').saveAsTable('bronze_crashes')
spark.sql('SELECT crash_year, count(*) cnt FROM bronze_crashes GROUP BY 1 ORDER BY 1').show()
print('DONE:', spark.table('bronze_crashes').count(), 'rows')

In [ ]:
# Schools (optional)
try:
    sr = requests.get('https://data.lacity.org/resource/qh44-jvjp.json?$limit=50000', timeout=60)
    sr.raise_for_status()
    schools = sr.json()
    if schools:
        keep = [k for k in schools[0].keys() if not isinstance(schools[0][k], (dict, list))]
        srows = [tuple(str(s.get(k, '') or '') for k in keep) for s in schools]
        from pyspark.sql.types import StructType, StructField, StringType
        sdf = spark.createDataFrame(srows, StructType([StructField(k, StringType()) for k in keep]))
        sdf.write.format('delta').mode('overwrite').saveAsTable('bronze_schools')
        print('Schools:', sdf.count())
except Exception as e:
    print('Schools skipped:', e)

In [ ]:
print('=== BRONZE INGEST COMPLETE ===')
print('bronze_crashes:', spark.table('bronze_crashes').count())
try:
    print('bronze_schools:', spark.table('bronze_schools').count())
except:
    print('bronze_schools: skipped')